# EOV-MMP pilot — Colab runner

Runs the diagnostic pilot for the H1'/H2 hypotheses. See
`pilot_analysis/report.md` in the repo for what is being measured and why.

**Before you start:** Runtime → Change runtime type → **T4 GPU** or better.
16 GB VRAM is the minimum; evaluation fits, training probably does not.

**What lives where.** Frames and weights come from two private HuggingFace
repos and land on the session disk (ephemeral, rebuilt each time). Results go
to Drive, because a dropped session must not cost you the run.

**The frames are never all on disk at once.** 7.48 GiB of per-video tars are
fetched a batch at a time, run, and deleted (`--disk-budget`). There is no
separate download step and no 30-40 minute frame decode.

**Run the cells in order.** Cell 4 (compiling the CUDA operator) is the
likeliest to fail on a fresh VM, which is why it is isolated.


## 1. Confirm the GPU


In [ ]:
!nvidia-smi
import torch
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))
cc = torch.cuda.get_device_capability()
print('compute capability', cc, '-> TORCH_CUDA_ARCH_LIST =', f'{cc[0]}.{cc[1]}')
gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'VRAM {gb:.1f} GiB usable')
# A T4 reports 15360 MiB and ~14.6 GiB usable after the driver reserve. That is
# expected and fine: the ~16 GB figure in the README is for TRAINING. Evaluation
# holds the two CLIP encoders (~2.7 GB), the end-to-end model (~2.5 GB) and ONE
# frame of activations -- end2end_model.py moves patches to the GPU a frame at a
# time, so the per-video tensors stay in CPU RAM (2.4 GiB for the longest test
# video, against Colab's ~12.7 GB).
assert gb > 13, 'under 13 GiB is too little even for evaluation; switch runtime'


## 2. Mount Drive and clone

If the GitHub repo is private, use
`https://<user>:<token>@github.com/...` instead.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%cd /content
!git clone -q https://github.com/ghibli613/ov-vidvrd-lab.git
%cd /content/ov-vidvrd-lab
!git log --oneline -1


In [ ]:
import os
# Data on the session disk (ephemeral). Output on Drive (survives a drop).
os.environ['VIDVRD_DATA_ROOT']   = '/content/data/vidvrd'
os.environ['VIDVRD_OUTPUT_ROOT'] = '/content/drive/MyDrive/vidvrd/output'
os.makedirs('/content/drive/MyDrive/vidvrd/preds', exist_ok=True)
!mkdir -p /content/preds
!python -m utils.paths


## 3. Dependencies

**Do not install PyTorch.** Colab ships a build matched to its driver;
replacing it is slow and usually breaks CUDA.


In [ ]:
!pip install -q ftfy regex einops timm fvcore pycocotools \
                opencv-python-headless gdown huggingface_hub


## 4. Compile the CUDA operator  ← most likely failure point

`MultiScaleDeformableAttention` builds from source. `7.5` is correct for a
T4; use the value cell 1 printed for anything else (`8.0` A100, `8.9` L4).

If it fails: delete `ops/build/` and retry. An `undefined symbol` on import
means it was built against a different torch.


In [ ]:
%cd /content/ov-vidvrd-lab/ops
!TORCH_CUDA_ARCH_LIST="7.5" python setup.py build install 2>&1 | tail -5
%cd /content/ov-vidvrd-lab


In [ ]:
import torch, MultiScaleDeformableAttention
print('operator imports OK')


## 5. HuggingFace token

Both repos are private. Add `HF_TOKEN` under the key icon in the left
sidebar, with notebook access enabled.


In [ ]:
from google.colab import userdata
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
from huggingface_hub import whoami
print('logged in as', whoami()['name'])


## 6. Small data: annotations, trajectories, class splits, GT

Downloads from public sources and builds the GT jsons locally, ~2 minutes.

**Do not add `frames` to `--steps`.** That is the 30-40 minute decode this
notebook exists to avoid; frames arrive per batch in cell 9.


In [ ]:
!python tools/prepare_data.py --steps anno,meta,gt


## 7. Weights — the eval subset only

`--only eval` fetches 2.92 GB and skips 4.5 GB of step-1/2/3 training
components you do not need for the pilot.


In [ ]:
!python tools/hugging_download.py --manifest https://huggingface.co/ghibli613/ov-vidvrd-weights/resolve/main/MANIFEST.json --only eval


In [ ]:
!ls -la output/ckpt/ && du -sh output/ckpt data/vidvrd/data


## 8. The stride question — run BOTH, then compare

`--frame_stride` decides how often a frame is CLIP-encoded; skipped frames
reuse the last encoded frame's features.

* The **paper** says frames are sampled every 30.
* The **released code** has that sampling commented out, so it encodes every
  frame (stride 1).
* Which one produced the released checkpoints is **unknown**.

This is a ~30x compute difference and it also decides whether a cached-feature
dataset is ever viable. Settle it on 5 videos before committing to 200.

Reproduction targets for this checkpoint: **all mAP 26.88 / novel 15.64**
(the author's own reported figures — not the paper's 26.34 / 15.04).


In [ ]:
# stride 30 -- the paper's reading, ~30x cheaper
!python pilot_analysis/scripts/dump_predictions.py \
    --ckpt_path output/ckpt/baseline_fbce_vidvrd_bs1_lr1e-05_dim512_none_rel_mot_clip_bbox_end2end_base-001.pth \
    --path_AFLink output/ckpt/AFLink_epoch20.pth \
    --shards https://huggingface.co/datasets/ghibli613/ov-vidvrd-frames/resolve/main/SHARDS.json \
    --out /content/preds/stride30 \
    --drive-copy /content/drive/MyDrive/vidvrd/preds/stride30 \
    --disk-budget 1.0 --frame_stride 30 --limit 5


In [ ]:
# stride 1 -- what the shipped code actually does
!python pilot_analysis/scripts/dump_predictions.py \
    --ckpt_path output/ckpt/baseline_fbce_vidvrd_bs1_lr1e-05_dim512_none_rel_mot_clip_bbox_end2end_base-001.pth \
    --path_AFLink output/ckpt/AFLink_epoch20.pth \
    --shards https://huggingface.co/datasets/ghibli613/ov-vidvrd-frames/resolve/main/SHARDS.json \
    --out /content/preds/stride1 \
    --drive-copy /content/drive/MyDrive/vidvrd/preds/stride1 \
    --disk-budget 1.0 --frame_stride 1 --limit 5


In [ ]:
import json
for s in ('stride30','stride1'):
    try:
        m = json.load(open(f'/content/preds/{s}/metrics.json'))
    except FileNotFoundError:
        print(f'{s}: no metrics.json'); continue
    for split in ('all','novel'):
        d = m.get(split, {})
        if 'mAP' in d:
            print(f"{s:9} {split:6} mAP {d['mAP']*100:6.2f}  "
                  f"R@50 {d['R@50']*100:6.2f}  R@100 {d['R@100']*100:6.2f}  "
                  f"{d['minutes']:.1f} min / {d['new_videos']} videos")


### 8b. Does stride 30 freeze the visual input?

Predicted from the code but never measured: at stride 30 the per-frame CLIP
features within a 30-frame block are *the same tensor*. The detector runs
per frame on those features and is deterministic, so its boxes should be
identical across the block — meaning trajectories become step functions and
there is no motion inside a segment.

If this prints `IDENTICAL`, the stride question is settled without needing
the mAP comparison at all.


In [ ]:
import os, torch
from utils.parser_func import parse_args
import sys

# one video's worth of frames must be on disk; fetch a single shard
from tools.hugging_download import load_manifest
sys.argv = ['x', '--frame_stride', '30']
args = parse_args()

from data_loading.dataset import Dataset_new
ds = Dataset_new(args, 'val')
vid = ds.path_list[0]
print('checking', vid)
if not os.path.isdir(os.path.join(os.environ['VIDVRD_DATA_ROOT'], 'frames', vid)):
    print('frames for this video are not on disk; run cell 9 first, or pick a')
    print('video whose batch has been fetched. Skipping.')
else:
    item = ds[0]
    p = item['patch_']
    same = [bool(torch.equal(p[0], p[i])) for i in range(1, min(30, p.shape[0]))]
    print('frames 2-30 identical to frame 1:', all(same))
    print('VERDICT:', 'IDENTICAL -- stride 30 freezes the visual input'
          if all(same) else 'DIFFERENT -- features vary within the block')


## 9. The full run

All 200 test videos, both predicate splits. Frames stream in a batch at a
time, so peak disk stays near `--disk-budget`.

Results flush every 5 videos and are mirrored to Drive. **If the session
drops, re-run this cell** — it resumes from what is already on disk.

Set `--frame_stride` to whichever value cell 8 justified.


In [ ]:
!python pilot_analysis/scripts/dump_predictions.py \
    --ckpt_path output/ckpt/baseline_fbce_vidvrd_bs1_lr1e-05_dim512_none_rel_mot_clip_bbox_end2end_base-001.pth \
    --path_AFLink output/ckpt/AFLink_epoch20.pth \
    --shards https://huggingface.co/datasets/ghibli613/ov-vidvrd-frames/resolve/main/SHARDS.json \
    --out /content/preds/full \
    --drive-copy /content/drive/MyDrive/vidvrd/preds/full \
    --disk-budget 1.0 --flush-every 5 \
    --frame_stride 30


### If the session dropped

The session disk is gone but Drive is not. Restore the partial results,
then re-run cell 9 — it skips every video already present.


In [ ]:
!mkdir -p /content/preds/full
!cp -n /content/drive/MyDrive/vidvrd/preds/full/*.json /content/preds/full/ 2>/dev/null || true
import json, os
p = '/content/preds/full/final_merged_all.json'
print('videos already done:', len(json.load(open(p))) if os.path.exists(p) else 0)


## 10. Check what landed on Drive


In [ ]:
!ls -la /content/drive/MyDrive/vidvrd/preds/full/
import json
m = json.load(open('/content/preds/full/metrics.json'))
print(json.dumps(m.get('config', {}), indent=1))
for split in ('all','novel'):
    d = m.get(split, {})
    if 'mAP' in d:
        print(f"{split:6} mAP {d['mAP']*100:6.2f}  R@50 {d['R@50']*100:6.2f}  "
              f"R@100 {d['R@100']*100:6.2f}   ({d['videos']} videos, "
              f"{d['instances']} instances)")
print()
print('Phase 1 gate: all 26.88 / novel 15.64 (this checkpoint), tolerance +-1.0')


## 10b. Phases 2 and 3 — the numbers the study turns on

CPU only, runs right here once the dumps exist. Produces, in one pass:

* **H1'** — `geometric_dynamic` vs `geometric_static` mAP on both splits,
  with the ≤0.5× verdict line
* **the granularity control** — per verb family, because the two groups hold
  78 vs 44 predicates and that alone could explain any gap
* **the mechanism check** — are novel-predicate errors *within* a verb family
  (`fly_front` → `fly_right`) as §B.11 predicted before any prediction existed?
* **H2** — oracle-merge gain, the relaxed-vIoU split between mislocalisation
  and lost content, and the loss rate (not fragmentation — see §C.5)

Save this output. It is what goes back to the chat conversation.


In [ ]:
!python pilot_analysis/scripts/phase2_phase3.py --preds /content/preds/full \
    2>&1 | tee /content/drive/MyDrive/vidvrd/preds/phase2_phase3_results.txt


### Copy the dumps to Drive

`segments_raw_*.json` is the expensive one — it carries the per-pair
trajectories that Phase 3 needs, and regenerating it means another GPU pass.


In [ ]:
!mkdir -p /content/drive/MyDrive/vidvrd/preds/full
!cp -v /content/preds/full/*.json /content/drive/MyDrive/vidvrd/preds/full/
!du -sh /content/drive/MyDrive/vidvrd/preds/full/


## 11. What to hand back

Two files, both on Drive:

* `preds/phase2_phase3_results.txt` — the H1'/H2 verdicts and every table
* `preds/full/metrics.json` — the Phase 1 reproduction numbers

Plus, from the notebook itself: the stride comparison in cell 8 and the
frozen-input verdict in cell 8b.

Read the results against the pre-registered predictions in
`pilot_analysis/PILOT-STATUS.md`:

* §B.11 — novel-predicate confusions cluster **within** a verb family
* §B.12 — the oracle-merge gain clears 2 mAP comfortably
* §C.2 — the appearance groups are too small to carry a verdict; report them
  descriptively, with n
* §C.5 — judge H2 on the loss rate, not the fragmentation rate

A result that contradicts a prediction is a finding, not a bug — but check the
input files first if H2's gain comes out negative (see the note at the top of
`phase2_phase3.py`).
